In [0]:
-- Lendo a tabela df_limpo
SELECT *
FROM `data-growth-analytic`.silver.df_limpo


In [0]:
create or replace table 
`data-growth-analytic`.gold.status_pedido AS
select  status_pedido AS pedido,
CAST(SUM(receita) AS DECIMAL(18,2)) AS receita_total

FROM `data-growth-analytic`.silver.df_limpo 
WHERE status_pedido IS NOT NULL
GROUP BY pedido
order by receita_total;

In [0]:
create or replace table 
`data-growth-analytic`.gold.PGTO AS
select  metodo_pagamento AS PGTO,
CAST(SUM(receita) AS DECIMAL(18,2)) AS receita_total

FROM `data-growth-analytic`.silver.df_limpo 
WHERE metodo_pagamento IS NOT NULL
GROUP BY PGTO
order by receita_total;

In [0]:
create or replace table 
`data-growth-analytic`.gold.regioes AS
select   DATE_FORMAT(data_pedido, 'yyyy-MM') AS mes,
 regiao, 
 CAST(SUM(receita) AS DECIMAL(18,2)) AS receita_total

FROM `data-growth-analytic`.silver.df_limpo 
WHERE data_pedido IS NOT NULL
GROUP BY mes, regiao;

In [0]:
create or replace table 
`data-growth-analytic`.gold.clientes AS
select  DATE_FORMAT(data_pedido, 'yyyy-MM') AS mes,
 id_cliente, 
 CAST(SUM(receita) AS DECIMAL(18,2)) AS receita_total

FROM `data-growth-analytic`.silver.df_limpo 
WHERE data_pedido IS NOT NULL
GROUP BY mes, id_cliente
order by mes;

In [0]:
create or replace table 
`data-growth-analytic`.gold.segmento AS
select   DATE_FORMAT(data_pedido, 'yyyy-MM') AS mes,
 segmento_cliente, 
 CAST(SUM(receita) AS DECIMAL(18,2)) AS receita_total

FROM `data-growth-analytic`.silver.df_limpo 
WHERE data_pedido IS NOT NULL
GROUP BY mes, segmento_cliente
order by mes,receita_total;

In [0]:
select * FROM `data-growth-analytic`.silver.df_limpo 

In [0]:
--create or replace table 
--`data-growth-analytic`.gold.quantidade AS
select   --DATE_FORMAT(data_pedido, 'yyyy-MM') AS mes,
 id_pedido,
 Sum(ticket) as qtde

FROM `data-growth-analytic`.silver.df_limpo 
WHERE data_pedido IS NOT NULL
GROUP BY id_produto
order by id_produto asc;

In [0]:
create or replace table 
`data-growth-analytic`.gold.quantidade AS
select   --DATE_FORMAT(data_pedido, 'yyyy-MM') AS mes,
 id_produto,
 Sum(quantidade) as qtde

FROM `data-growth-analytic`.silver.df_limpo 
WHERE data_pedido IS NOT NULL
GROUP BY id_produto
order by id_produto asc;

In [0]:
CREATE OR REPLACE TABLE
`data-growth-analytic`.gold.metricas AS
SELECT
    COUNT(*) AS total_linhas,
    COUNT(DISTINCT id_pedido) AS pedidos_unicos,
    MIN(receita) AS menor_receita,
    MAX(receita) AS maior_receita,
    SUM(receita) AS receita_total
FROM `data-growth-analytic`.silver.df_limpo
WHERE id_pedido IS NOT NULL;

In [0]:
CREATE OR REPLACE TABLE
`data-growth-analytic`.gold.kpi_receita AS

SELECT
    DATE_FORMAT(data_pedido, 'yyyy-MM') AS mes,

    COUNT(DISTINCT id_pedido) AS total_pedidos,

    CAST(SUM(receita) AS DECIMAL(18,2)) AS receita_total,

    CAST(
        SUM(receita) / COUNT(DISTINCT id_pedido)
        AS DECIMAL(18,2)
    ) AS ticket_medio

FROM `data-growth-analytic`.silver.df_limpo

WHERE
    id_pedido IS NOT NULL
    AND data_pedido IS NOT NULL

GROUP BY
    DATE_FORMAT(data_pedido, 'yyyy-MM')

ORDER BY mes;

In [0]:
DESCRIBE TABLE `data-growth-analytic`.gold.kpi_receita;

In [0]:
select * From `data-growth-analytic`.gold.kpi_receita

In [0]:
SELECT
    SUM(CAST(receita AS DECIMAL(18,2))) AS receita_total
FROM `data-growth-analytic`.silver.df_limpo;

In [0]:
-- Lendo a tabela df_limpo
SELECT data_cadastro,data_envio,data_pedido, receita
FROM `data-growth-analytic`.silver.df_limpo

In [0]:
SELECT convertido, clicado, enviado, aberto
FROM `data-growth-analytic`.silver.df_limpo

In [0]:
CREATE OR REPLACE TABLE
`data-growth-analytic`.gold.kpi_receita_taxas AS

SELECT
    DATE_FORMAT(data_pedido, 'yyyy-MM') AS mes,
    nome_campanha,
    SUM(convertido) * 100.0 /  NULLIF(COUNT(*),0) AS taxa_conversao,
    SUM(clicado) * 100.0 /  NULLIF(SUM(enviado), 0) AS taxa_clique,
    SUM(enviado) * 100.0 /  NULLIF(COUNT(*), 0) AS taxa_envio,
    SUM(aberto) * 100.0 /  NULLIF(SUM(enviado),0) AS taxa_abertura,
    SUM(convertido) * 100.0 /  NULLIF(SUM(aberto),0) AS taxa_conversao_por_abertura,
    SUM(convertido) * 100.0 /  NULLIF(SUM(clicado),0) AS taxa_conversao_por_clique,
    SUM(aberto) * 100.0 /  NULLIF(SUM(enviado),0) AS taxa_abertura_por_envio

FROM `data-growth-analytic`.silver.df_limpo

WHERE id_pedido IS NOT NULL

GROUP BY mes, nome_campanha;

In [0]:
select canal_campanha FROM `data-growth-analytic`.silver.df_limpo

In [0]:
CREATE OR REPLACE TABLE
`data-growth-analytic`.gold.kpi_resultados_crm AS
SELECT
    COUNT(*) AS total,

    COUNT(canal_acesso) AS canal_acesso_preenchido,
    COUNT(canal_campanha) AS canal_campanha_preenchido,
    COUNT(nome_campanha) AS campanha_preenchida,

    SUM(enviado) AS enviados,
    SUM(aberto) AS abertos,
    SUM(clicado) AS cliques,
    SUM(convertido) AS conversoes

FROM `data-growth-analytic`.silver.df_limpo;

In [0]:
CREATE OR REPLACE TABLE
`data-growth-analytic`.gold.kpi_canal_acesso AS
SELECT
    canal_acesso,
    COUNT(*) AS total_registros,
    SUM(convertido) AS conversoes

FROM `data-growth-analytic`.silver.df_limpo

GROUP BY canal_acesso

ORDER BY conversoes DESC;

In [0]:
CREATE OR REPLACE TABLE
`data-growth-analytic`.gold.kpi_canal_por_cliente AS
SELECT
    canal_acesso,

    COUNT(DISTINCT id_cliente) AS total_clientes,

    SUM(convertido) AS conversoes,

    ROUND(
        100.0 * SUM(convertido) /
        NULLIF(COUNT(DISTINCT id_cliente), 0),
        2
    ) AS taxa_conversao

FROM `data-growth-analytic`.silver.df_limpo

GROUP BY canal_acesso

ORDER BY taxa_conversao DESC;

In [0]:
SELECT
    customer_id,
    campaign_name,
    campaign_channel,
    sent,
    opened,
    clicked,
    converted,
    sent_date
FROM `data-growth-analytic`.bronze.rascunho
LIMIT 20;

In [0]:
CREATE OR REPLACE TABLE
`data-growth-analytic`.gold.kpi_canal_campanha AS

SELECT
    canal_campanha,
    SUM(convertido) * 100.0 /  NULLIF(COUNT(*),0) AS taxa_conversao,
    SUM(clicado) * 100.0 /  NULLIF(SUM(enviado), 0) AS taxa_clique,
    SUM(enviado) * 100.0 /  NULLIF(COUNT(*), 0) AS taxa_envio,
    SUM(aberto) * 100.0 /  NULLIF(SUM(enviado),0) AS taxa_abertura,
    SUM(convertido) * 100.0 /  NULLIF(SUM(aberto),0) AS taxa_conversao_por_abertura,
    SUM(convertido) * 100.0 /  NULLIF(SUM(clicado),0) AS taxa_conversao_por_clique,
    SUM(aberto) * 100.0 /  NULLIF(SUM(enviado),0) AS taxa_abertura_por_envio


FROM `data-growth-analytic`.silver.df_limpo
Where canal_campanha is not null
GROUP BY canal_campanha;

In [0]:
select * from `data-growth-analytic`.gold.kpi_receita_taxas

In [0]:
select * from `data-growth-analytic`.gold.kpi_receita_taxas

In [0]:
CREATE OR REPLACE TABLE
`data-growth-analytic`.gold.kpi_canal_nome_campanha AS

SELECT
    nome_campanha,
    SUM(convertido) * 100.0 /  NULLIF(COUNT(*),0) AS taxa_conversao,
    SUM(clicado) * 100.0 /  NULLIF(SUM(enviado), 0) AS taxa_clique,
    SUM(enviado) * 100.0 /  NULLIF(COUNT(*), 0) AS taxa_envio,
    SUM(aberto) * 100.0 /  NULLIF(SUM(enviado),0) AS taxa_abertura,
    SUM(convertido) * 100.0 /  NULLIF(SUM(aberto),0) AS taxa_conversao_por_abertura,
    SUM(convertido) * 100.0 /  NULLIF(SUM(clicado),0) AS taxa_conversao_por_clique,
    SUM(aberto) * 100.0 /  NULLIF(SUM(enviado),0) AS taxa_abertura_por_envio


FROM `data-growth-analytic`.silver.df_limpo

Where nome_campanha IS NOT NULL

GROUP BY nome_campanha;

In [0]:
select * from `data-growth-analytic`.silver.df_limpo

In [0]:
CREATE OR REPLACE TABLE
`data-growth-analytic`.gold.kpi_cancelamento AS

select count(data_pedido),count(data_envio)
from `data-growth-analytic`.silver.df_limpo
where data_pedido is not null

In [0]:
CREATE OR REPLACE TABLE
`data-growth-analytic`.gold.kpi_por_idade AS
select idade, 
    SUM(CAST(receita AS DECIMAL(18,2))) AS receita_total
from `data-growth-analytic`.silver.df_limpo
where idade is not null
group by idade